# Atividade Prática — Aula 3: Limpeza, Preparação e Qualidade dos Dados com Pandas

Esta atividade foi construída com base nos slides da Aula 3, que destacam a **limpeza como a fundação invisível** de qualquer dashboard confiável. O objetivo aqui não é apenas "fazer funcionar", mas tomar decisões conscientes sobre tipos, valores ausentes, duplicidades, variáveis derivadas e exportação da base limpa. fileciteturn2file0

## Regras desta atividade
- Você deve **construir os códigos**.
- O notebook orienta os passos, mas não entrega as soluções prontas.
- Sempre que fizer uma decisão de limpeza, **documente o porquê** em comentário ou célula markdown.
- Ao final, exporte uma base limpa para uso nas próximas aulas.

## Dataset desta atividade
Arquivo bruto: `vendas_brasil_aula3_bruto.csv`


## 1. Preparação do ambiente

Importe as bibliotecas necessárias para trabalhar com:
- leitura de dados
- tratamento de nulos
- identificação de infinitos
- exportação do resultado final

**Sugestão:** Pandas e NumPy já resolvem toda a atividade.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## 2. Leitura da base bruta

Leia o arquivo `vendas_brasil_aula3_bruto.csv` em um DataFrame chamado `df`.

Depois:
1. Exiba as primeiras linhas
2. Exiba as últimas linhas
3. Observe visualmente se já existem sinais de sujeira


In [ ]:
df = pd.read_csv('vendas_brasil_aula3_bruto.csv')
df.head()

## 3. Check-up inicial do dataset

Com base no checklist de um dataset "clean" apresentado na aula, faça um diagnóstico inicial da base. fileciteturn2file0

### Sua missão
Use comandos que permitam responder:
- Qual é o tamanho da base?
- Quais são os tipos atuais das colunas?
- Existem valores nulos?
- Há colunas com tipo inadequado?
- Há algo que pareça impedir análises confiáveis?


In [ ]:
print("Shape:", df.shape)
print("\nTipos de dados:")
print(df.dtypes)

print("\nValores ausentes:")
print(df.isna().sum())

print("\nDuplicidades exatas:", df.duplicated().sum())
print("Duplicidades por pedido_id:", df['pedido_id'].duplicated().sum())

print("\nAmostra de categorias:")
print("UF:")
print(df['uf'].value_counts(dropna=False))
print("\nCanal:")
print(df['canal'].value_counts(dropna=False))
print("\nCategoria:")
print(df['categoria'].value_counts(dropna=False))

## 4. Batalha 1 — A tirania dos tipos de dados

Nos slides, vimos que datas não podem ser tratadas como texto e que números em formato string quebram análises. fileciteturn2file0

### Tarefa
Converta corretamente, quando fizer sentido:
- `data`
- `receita`
- `lucro`
- `quantidade`

### Orientação
- Investigue valores estranhos antes de converter
- Pense no uso de `errors='coerce'`
- Registre em markdown ou comentário quais problemas encontrou


In [ ]:
# Conversão da data
df['data_dt'] = pd.to_datetime(df['data'], format='mixed', errors='coerce', dayfirst=True)

# Conversão da receita para número
def clean_receita(val):
    if pd.isna(val):
        return np.nan
    s = str(val).strip()
    if s.lower() == 'zero':
        return 0.0
    s = s.replace('R$', '').replace(' ', '')
    if ',' in s and '.' in s:
        s = s.replace('.', '').replace(',', '.')
    elif ',' in s:
        s = s.replace(',', '.')
    try:
        return float(s)
    except:
        return np.nan

df['receita_num'] = df['receita'].apply(clean_receita)

print("Datas inválidas:", df['data_dt'].isna().sum())
print("Receitas não convertidas:", df['receita_num'].isna().sum())
print("\nLinhas com data inválida:")
print(df[df['data_dt'].isna()][['pedido_id', 'data']])

print("\nLinhas com receita inválida:")
print(df[df['receita_num'].isna()][['pedido_id', 'receita']].head())

### Reflexão curta
Explique:
1. Por que deixar `data` como texto pode quebrar análises temporais?
2. Por que deixar `receita` como string pode distorcer cálculos?


### Resposta - Reflexão curta

1. **Deixar `data` como texto** dificulta ordenar corretamente por período, agrupar por mês, calcular intervalos e construir análises temporais confiáveis. Datas em texto podem ser interpretadas de forma errada ou nem serem reconhecidas.

2. **Deixar `receita` como string** distorce somas, médias, rankings e margens, porque o Pandas pode tratar os valores como texto em vez de número. Isso quebra cálculos e pode gerar dashboards incorretos.

## 5. Batalha 2 — O enigma dos valores ausentes

A aula reforça que valores ausentes não devem ser tratados automaticamente; a decisão precisa ser justificada. fileciteturn2file0

### Tarefa
1. Mapeie os valores ausentes por coluna
2. Identifique quais colunas críticas têm nulos
3. Defina uma estratégia para cada caso:
   - remover linhas?
   - preencher?
   - manter?
4. Justifique cada escolha

### Dica
Evite decisões automáticas sem análise de contexto.


In [ ]:
# Cópia para tratamento
df_tratado = df.copy()

# Preenchimentos transparentes
df_tratado['observacao'] = df_tratado['observacao'].fillna('sem observacao')
df_tratado['uf'] = df_tratado['uf'].replace({'mg': 'MG', 'rj': 'RJ'}).fillna('Não informado')
df_tratado['canal'] = df_tratado['canal'].replace({
    'ONLINE': 'Online',
    'Loja fisica': 'Loja Física',
    'MarketPlace': 'Marketplace'
}).fillna('Não informado')
df_tratado['categoria'] = df_tratado['categoria'].replace({
    'telefonia': 'Telefonia',
    'Perifericos': 'Periféricos'
}).fillna('Não informado')

# Remoção de linhas com falhas críticas
antes = len(df_tratado)
df_tratado = df_tratado.dropna(subset=['data_dt', 'receita_num', 'lucro']).copy()
depois = len(df_tratado)

print("Linhas removidas por data/receita/lucro ausentes ou inválidos:", antes - depois)
print("\nNulos após tratamento:")
print(df_tratado.isna().sum())

### Registro reflexivo
Escreva aqui sua justificativa para o tratamento dos nulos:
- Em quais colunas você removeu linhas?
- Em quais colunas você preencheu valores?
- Em quais situações decidiu manter o nulo?


### Resposta - Registro reflexivo

- **Removi linhas** quando havia falhas críticas em `data_dt`, `receita_num` ou `lucro`, porque essas colunas são essenciais para análise temporal e financeira.
- **Preenchi valores** em `observacao` com `sem observacao` e nas colunas categóricas `uf`, `canal` e `categoria` com `Não informado` quando necessário, além de padronizar grafias como `mg` → `MG` e `ONLINE` → `Online`.
- **Mantive nulo** apenas na `margem_lucro_pct` quando a `receita_num` era zero, porque nesse caso a margem não deve ser calculada artificialmente.

## 6. Batalha 3 — O ataque dos clones (duplicidades)

A aula alerta que nem toda duplicidade é erro automático: pode haver compra legítima repetida ou dupla inserção do sistema. fileciteturn2file0

### Tarefa
1. Investigue se há linhas duplicadas
2. Analise se a duplicidade parece nociva para os KPIs
3. Escolha uma estratégia:
   - remover duplicidades totais?
   - remover apenas com base em certas colunas?
   - manter?
4. Justifique a decisão


In [ ]:
print("Duplicidades exatas antes:", df_tratado.duplicated().sum())
print("Duplicidades por pedido_id antes:", df_tratado['pedido_id'].duplicated().sum())

# Remover duplicidades exatas
df_tratado = df_tratado.drop_duplicates().copy()

print("\nDuplicidades exatas depois:", df_tratado.duplicated().sum())
print("Duplicidades por pedido_id depois:", df_tratado['pedido_id'].duplicated().sum())

## 7. Padronização de categorias

Os slides mostram que categorias duplicadas ou mal escritas podem distorcer rankings e filtros. fileciteturn2file0

### Tarefa
Inspecione e padronize, se necessário:
- `uf`
- `canal`
- `categoria`

### Pense em:
- maiúsculas e minúsculas
- espaços extras
- acentuação / variações
- categorias equivalentes escritas de formas diferentes


In [ ]:
# Padronização final das categorias
df_tratado['uf'] = df_tratado['uf'].str.upper()
df_tratado['canal'] = df_tratado['canal'].replace({
    'ONLINE': 'Online',
    'Loja fisica': 'Loja Física',
    'MarketPlace': 'Marketplace'
})
df_tratado['categoria'] = df_tratado['categoria'].replace({
    'telefonia': 'Telefonia',
    'Perifericos': 'Periféricos'
})

print("UF padronizada:")
print(df_tratado['uf'].value_counts(dropna=False))

print("\nCanal padronizado:")
print(df_tratado['canal'].value_counts(dropna=False))

print("\nCategoria padronizada:")
print(df_tratado['categoria'].value_counts(dropna=False))

## 8. Subindo de nível — Criação de variáveis derivadas

Depois da limpeza, é hora de enriquecer a base com variáveis úteis para análise. fileciteturn2file0

### Tarefa
Crie, se possível:
- `ano`
- `mes`
- `ano_mes`
- `margem_lucro`

### Atenção
A criação de `margem_lucro` exige cuidado com divisões por zero e possíveis valores infinitos.


In [ ]:
# Variáveis derivadas
df_tratado['ano'] = df_tratado['data_dt'].dt.year
df_tratado['mes'] = df_tratado['data_dt'].dt.to_period('M').astype(str)
df_tratado['margem_lucro_pct'] = np.where(
    df_tratado['receita_num'] > 0,
    (df_tratado['lucro'] / df_tratado['receita_num']) * 100,
    np.nan
)

df_tratado[['pedido_id', 'data_dt', 'receita_num', 'lucro', 'ano', 'mes', 'margem_lucro_pct']].head()

### Reflexão técnica
Explique:
1. O que pode acontecer ao calcular margem de lucro quando a receita é zero?
2. Como você decidiu tratar esse caso?


### Resposta - Reflexão técnica

1. Quando a **receita é zero**, o cálculo da margem de lucro pode gerar divisão por zero, produzindo infinito, erro ou um valor sem sentido analítico.

2. Nesse caso, optei por definir a **margem como `NaN`** quando `receita_num <= 0`, porque isso evita distorções e sinaliza claramente que aquela linha não é adequada para esse cálculo.

## 9. Seleção final — Menos é mais

A aula propõe que nem toda coluna precisa seguir para a base analítica final. fileciteturn2file0

### Tarefa
Crie um DataFrame final, por exemplo `df_clean` ou `df_dash`, contendo apenas as colunas estritamente necessárias para análises de negócio.

**Sugestão de raciocínio:**
- Quais colunas ajudam a responder perguntas de negócio?
- Quais colunas são operacionais, auxiliares ou dispensáveis?
- O que precisa existir para as próximas aulas de visualização e dashboard?


In [ ]:
colunas_finais = [
    'pedido_id', 'data_dt', 'ano', 'mes', 'uf', 'canal', 'categoria',
    'produto', 'cliente_id', 'quantidade', 'receita_num', 'lucro',
    'margem_lucro_pct', 'observacao'
]

df_clean = df_tratado[colunas_finais].copy()
df_clean.head()

## 10. Validação final da base limpa

Antes de exportar, faça uma checagem final.

### Verifique:
- os tipos estão corretos?
- ainda há nulos problemáticos?
- ainda há duplicidades nocivas?
- existe algum infinito em `margem_lucro`?
- a base está pronta para ser usada em análises e dashboards?


## Resumo dos principais ajustes realizados

- Conversão de `data` para formato de data
- Conversão de `receita` para valor numérico
- Remoção de linhas com falhas críticas em data, receita ou lucro
- Tratamento de nulos em `observacao`
- Padronização de `uf`, `canal` e `categoria`
- Remoção de duplicidades exatas
- Criação das variáveis `ano`, `mes` e `margem_lucro_pct`

### Resultado final
A base final ficou pronta para análise, com estrutura mais confiável para dashboards e agrupamentos.


In [ ]:
print("Shape final:", df_clean.shape)
print("\nNulos finais:")
print(df_clean.isna().sum())

print("\nDuplicidades finais:", df_clean.duplicated().sum())
print("Duplicidades por pedido_id:", df_clean['pedido_id'].duplicated().sum())

print("\nTipos finais:")
print(df_clean.dtypes)

## 11. Exportação da base "Clean"

Agora gere o arquivo final da base tratada.

### Tarefa
Exporte sua base limpa com o nome:

`vendas_brasil_aula3_clean.csv`

### Observação
Esse arquivo será o selo de qualidade do seu trabalho desta aula.


In [ ]:
df_clean.to_csv('vendas_brasil_aula3_clean.csv', index=False)
print("Arquivo exportado com sucesso: vendas_brasil_aula3_clean.csv")

## 12. Conclusão e registro reflexivo

Escreva um pequeno texto respondendo:

1. Quais foram os principais problemas de qualidade encontrados?
2. Qual decisão de limpeza foi mais difícil?
3. Que impacto essas falhas poderiam causar em um dashboard?
4. Por que a etapa de limpeza é considerada a "fundação invisível" do projeto?


### Resposta - Conclusão e registro reflexivo

Os principais problemas de qualidade encontrados foram: datas em formatos mistos e inválidos, valores de receita armazenados como texto, valores ausentes em colunas importantes, categorias com grafias diferentes e duplicidades.

A decisão de limpeza mais delicada foi escolher entre preencher ou remover valores ausentes. Para colunas críticas de análise financeira e temporal, a melhor decisão foi remover as linhas problemáticas. Já em colunas descritivas, o preenchimento controlado foi mais adequado.

Essas falhas poderiam causar impactos sérios em um dashboard, como somas erradas, filtros quebrados, rankings distorcidos e interpretações equivocadas sobre desempenho por canal, UF ou período.

A limpeza é chamada de **fundação invisível** porque quase nunca aparece no gráfico final, mas é ela que garante que toda a análise acima dela seja confiável.

## 13. Desafio extra (opcional)

Use sua base limpa para responder rapidamente:

- Qual UF concentra maior receita?
- Qual canal gera maior lucro?
- Existe algum mês com desempenho claramente acima dos demais?

Você pode responder com tabelas simples ou pequenos agrupamentos.


In [ ]:
receita_por_uf = df_clean.groupby('uf')['receita_num'].sum().sort_values(ascending=False)
lucro_por_canal = df_clean.groupby('canal')['lucro'].sum().sort_values(ascending=False)
receita_por_mes = df_clean.groupby('mes')['receita_num'].sum().sort_values(ascending=False)

print("UF com maior receita:")
print(receita_por_uf.head())

print("\nCanal com maior lucro:")
print(lucro_por_canal)

print("\nMeses com maior receita:")
print(receita_por_mes.head())